# Deep text models on a Colab GPU

Runs `python -m cres deep` for MiniLM and DistilBERT on one window and copies the prediction
files back to Drive. All logic lives in `src/cres/deep.py`; the steps, including what to upload
first and where the outputs go, are in `notebooks/README.md`.

Runtime: **Runtime > Change runtime type > T4 GPU** (or better), then run the cells in order.

In [ ]:
import sys

WINDOW = "W5"
DRIVE = "/content/drive/MyDrive/cres_deep"
REPO = "https://github.com/omvyas77/Credit-risk-early-signals.git"

assert (3, 12) <= sys.version_info[:2] <= (3, 13), f"needs Python 3.12 or 3.13: {sys.version}"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!ls -l {DRIVE}/deep_input

In [ ]:
!test -d /content/cres || git clone --depth 1 --branch v2 {REPO} /content/cres
%cd /content/cres
!git log -1 --format="%h %s"
!pip install -q ".[deep]"

In [ ]:
!mkdir -p data
!cp -r {DRIVE}/deep_input data/
!du -sh data/deep_input

MiniLM: embeds every train, val and test narrative of the window (about 15 minutes on a T4),
then fits the two LRs on the CPU (up to 30 minutes). Its files are copied to Drive as soon as
it finishes.

In [ ]:
!python -m cres deep --model minilm --window {WINDOW}
!mkdir -p {DRIVE}/output/data/preds {DRIVE}/output/models
!cp data/preds/minilm__*__{WINDOW}.parquet {DRIVE}/output/data/preds/
!cp models/minilm__*__{WINDOW}.json {DRIVE}/output/models/

DistilBERT: fine-tunes on the training subsample, scores validation after each epoch, keeps
the best epoch, scores test, then stacks with the structured fields. If the runtime restarts,
rerun the cells above and this one: it resumes after the last finished epoch as long as the
runtime's disk survived.

In [ ]:
!python -m cres deep --model distilbert --window {WINDOW}
!mkdir -p {DRIVE}/output/data/preds {DRIVE}/output/models
!cp data/preds/distilbert__*__{WINDOW}.parquet {DRIVE}/output/data/preds/
!cp models/distilbert__*__{WINDOW}.json {DRIVE}/output/models/

In [ ]:
!ls -l {DRIVE}/output/data/preds {DRIVE}/output/models